# Multiprocessing no PyTorch (sem DDP)

Aula 3 - `mp` só sobe processos. Sem a toolbox, são **N treinos**, não um.
O `wsum` no fim não coincide: cada processo treinou o seu modelo.

## Por que multiprocessing existe

Um processo Python usa um núcleo (GIL). `torch.multiprocessing` cria filhos para usar os outros.

No DL: `torch.utils.data.DataLoader(num_workers=...)` (I/O) ou N jobs independentes (**N modelos**).
Não sincroniza gradiente - isso é a toolbox + DDP.

# 0. Instalando

`torchvision` só para o FashionMNIST.

In [15]:
%pip install -q torch torchvision

Note: you may need to restart the kernel to use updated packages.


## 1. Imports

`torch.multiprocessing` cria processos. `Queue` devolve o log ao notebook.
Sem `torch.distributed`.

In [16]:
import os

import torch
import torchvision


def pick_device(rank, world_size):
    """CUDA se existir; 2 processos compartilham a GPU se só houver uma."""
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        idx = rank if n >= world_size else 0
        torch.cuda.set_device(idx)
        return torch.device(f"cuda:{idx}")
    return torch.device("cpu")

## 2. Dois processos, sem treino

`torch.multiprocessing.start_processes` chama `hello` duas vezes (`rank=0` e `1`).
Prints intercalados são normais.

In [17]:
def hello(rank, world_size):
    """Sobe um processo e imprime pid + device. Sem treino."""
    device = pick_device(rank, world_size)
    print(f"rank {rank}/{world_size}  pid={os.getpid()}  device={device}")

torch.multiprocessing.start_processes(
    hello, args=(2,), nprocs=2, join=True, start_method="fork"
)

rank 0/2  pid=13081  device=cuda:0
rank 1/2  pid=13083  device=cuda:0


## 3. Modelo

MLP 28×28 → 10 classes. Cada processo cria a **sua** cópia.

In [18]:
class NeuralNetwork(torch.nn.Module):
    """MLP 28x28 -> 10 classes. Cada processo instancia a sua cópia."""
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(28 * 28, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)

## 4. Dataset por processo

Download no pai (última célula). Cada rank pega metade dos índices. Sem soma de gradiente.
`n_samples=2048` recorta o set: 60k imagens deixa a etapa 6 lenta.

In [19]:
def get_loader(rank, world_size, n_samples=2048):
    """Fatia o FashionMNIST por rank. Recorte pequeno: o set inteiro trava a aula."""
    data = torchvision.datasets.FashionMNIST(
        root="data",
        train=True,
        download=False,
        transform=torchvision.transforms.ToTensor(),
    )
    idx = list(range(rank, min(len(data), n_samples), world_size))
    return torch.utils.data.DataLoader(
        torch.utils.data.Subset(data, idx), batch_size=32, shuffle=True
    )

## 5. Treino local

Loop normal, sem DDP. `manual_seed(rank)` de propósito. `wsum` diferente = dois modelos.
CUDA se existir; senão CPU.

In [20]:
def train_one(rank, world_size, queue, epochs=2):
    """Treino local, sem all_reduce. O wsum no fim prova que são dois modelos."""
    torch.manual_seed(rank)
    device = pick_device(rank, world_size)
    loader = get_loader(rank, world_size)
    model = NeuralNetwork().to(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.001)
    loss_f = torch.nn.CrossEntropyLoss()
    loss = None
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_f(model(x), y)
            loss.backward()
            opt.step()
    wsum = sum(p.detach().sum().item() for p in model.parameters())
    queue.put((rank, os.getpid(), str(device), wsum, float(loss.detach())))